# 2 Navigating the filesystem

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part I — Orientation</span>
    <span class="bp-meta">Notebook&nbsp;2</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    The filesystem as a tree, and how to move through it: where you are, how to
    get elsewhere with paths, and how to see what is around you.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v0.1.0&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: find the repo root (the dir holding tools/check.sh), source the
# validation gate, and stand there so every relative path below is reckoned from
# the same place. In the live Binder terminal this same spot is your home (~).
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
cd "$ROOT"
# Give the data files fixed timestamps so `ls -t` orders them the same on every
# rebuild (git does not preserve modification times). Newest is .dataset-notes.
touch -d "2023-03-01 09:00" data/inputs data/inputs/* 2>/dev/null
touch -d "2023-04-01 09:00" data/trajectories data/trajectories/* 2>/dev/null
touch -d "2023-05-01 09:00" data/results data/results/* 2>/dev/null
touch -d "2023-02-01 09:00" data/logs data/logs/* 2>/dev/null
touch -d "2023-06-01 09:00" data/README.md 2>/dev/null
touch -d "2023-06-15 09:00" data/.dataset-notes 2>/dev/null

## What this notebook is about

In Notebook 1 you learned to run a command and to find help on your own. But a
command always acts *somewhere* — it lists *this* folder, opens *that* file. So
the next question is **where**: where you are in the filesystem, and how to get
to where the file you want lives.

The filesystem is a tree of folders, and this notebook is about walking it. Our
playground is the course's real `data/` folder — `.xyz` trajectories and
simulation inputs from the Molecular and Materials Modelling course. **You do not
need to know what any of these files are.** They are just files in folders; the
skill is reaching them, and that skill is identical whether the file holds atomic
coordinates or a grocery list.

## The filesystem is a tree

Every file on the machine lives in a directory (a "folder"), directories nest
inside other directories, and the whole thing hangs off a single top called the
**root**, written `/`. Follow the branches down and you reach every file; the
list of branches you followed, joined by slashes, is the file's **path** — its
address in the tree.

Two landmarks you will lean on constantly: `/` is the root of everything, and
**`~`** is *your home directory*, the branch you start on and where your own
files live. Here is the patch of tree we are about to explore:

<div style="background:#1b2233;border:1px solid #0e1422;border-radius:8px;padding:8px 20px 16px;margin:18px 0;">
  <div style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:.12em;text-transform:uppercase;color:#c0851a;font-weight:600;margin:8px 0 6px;">The course data, as a tree</div>
  <pre style="background:transparent;color:#cdd2db;margin:0;font-family:'JetBrains Mono',monospace;font-size:0.84rem;line-height:1.5;">/                              the root — everything is under here
└── ~  <span style="color:#8aa0b4;">(your home directory)</span>      where you start
    └── data/                  the playground
        ├── trajectories/
        │   ├── lj38-optimization.xyz
        │   └── lj38-relaxed.xyz
        ├── inputs/
        │   ├── geo-opt.inp
        │   ├── production-md.inp
        │   └── replica-exchange.lammps
        ├── results/
        │   ├── sn2-neb.ener
        │   └── pt-slab.xyz
        ├── logs/
        │   ├── gr2hno3-nvt.log
        │   └── gr2hno3-restart.log
        └── .dataset-notes     <span style="color:#8aa0b4;">(hidden — see ls -a, below)</span></pre>
</div>

## Where am I — `pwd` and the prompt

When you are unsure where you stand, ask. `pwd` ("print working directory") prints
the absolute path of the directory you are currently in:

In [2]:
pwd

/home/runner/work/bash-primer/bash-primer


That is the **working directory** — the folder every relative command is measured
from. There is a quieter way to know it, too: the **prompt itself can show your
location.** In this course's live terminal the prompt is set to display the
working directory, so it reads something like `~/data $` — the `$` still means
"your turn", but the part before it tells you where you are.

```{admonition} See it for yourself
:class: tip
The rendered cells on this page keep a plain `$` prompt, because a printed page
cannot know where *you* are. To watch the prompt track your location as you move,
open the [live terminal](https://mybinder.org/v2/gh/ramador09/bash-primer-public/HEAD?urlpath=shell/)
and run a few `cd` commands — the path before the `$` changes with every step.
```

## Moving around — `cd`

To go somewhere, use `cd` ("change directory") and give it the path of where you
want to be.

```{command-card} cd
```

Let us walk down into the data and back up. `cd` prints nothing when it succeeds —
just like a real terminal, the only sign it worked is that the next `pwd` reports
the new location:

In [3]:
cd data

In [4]:
pwd

/home/runner/work/bash-primer/bash-primer/data


In [5]:
cd ..

In [6]:
pwd

/home/runner/work/bash-primer/bash-primer


The `..` meant "up one level", so we landed back where we started. That little
`..` is one of a handful of path shorthands worth knowing cold — which brings us
to the heart of the notebook.

## Absolute vs relative paths

There are two ways to name any place in the tree, and the difference is the single
most useful idea in this notebook.

- An **absolute path** starts from the root `/` and spells out every branch:
  `/home/you/data/inputs/geo-opt.inp`. It means the same thing no matter where you
  are standing — it is the file's full address.
- A **relative path** starts from *where you are now* (your working directory):
  from the repo root, the same file is just `data/inputs/geo-opt.inp`. Shorter,
  but its meaning depends on where you stand.

Three shorthands appear in relative paths constantly: **`.`** is "here" (the
current directory), **`..`** is "up one level" (the parent), and **`~`** is your
home directory.

Here is the same file reached both ways. First relatively, from where we are:

In [7]:
ls data/results/sn2-neb.ener

data/results/sn2-neb.ener


Now the absolute path to the very same file — `$(pwd)` fills in the full address
of where we are, and we tack the rest on:

In [8]:
ls "$(pwd)/data/results/sn2-neb.ener"

/home/runner/work/bash-primer/bash-primer/data/results/sn2-neb.ener


Same file, two names. Use a relative path when the thing is near you (less to
type); reach for an absolute path when you need to name a place unambiguously,
from anywhere.

## Listing — `ls`

You have met `ls` already; now meet its workhorse flags. They turn a bare list of
names into something you can actually read — sizes, dates, hidden files, and more.

```{command-card} ls
```

A one-line motivation, then the physics-optional release valve: the `data/` files
are simulation outputs, and a real reason to list them is to see which run
produced the most data. You do not need to know what the files *are* — we are only
reading their **names, sizes, and dates**. Plain `ls` just names them:

In [9]:
ls data/trajectories

lj38-optimization.xyz  lj38-relaxed.xyz


Add `-l` for the long form — one file per line, with permissions, size (in bytes),
and a timestamp:

In [10]:
ls -l data/trajectories

total 160


-rw-r--r-- 1 runner runner 158697 Apr  1  2023 lj38-optimization.xyz


-rw-r--r-- 1 runner runner   2517 Apr  1  2023 lj38-relaxed.xyz


Those byte counts are hard to eyeball. Flags combine, so add `-h` for
human-readable sizes:

In [11]:
ls -lh data/trajectories

total 160K


-rw-r--r-- 1 runner runner 155K Apr  1  2023 lj38-optimization.xyz


-rw-r--r-- 1 runner runner 2.5K Apr  1  2023 lj38-relaxed.xyz


Now the size difference jumps out: one trajectory is far larger than the other.
And remember the gotcha from the card — without `-a`, the hidden `.dataset-notes`
file is invisible:

In [12]:
ls -a data

.  ..  .dataset-notes  README.md  inputs  logs	results  trajectories


There it is, alongside `.` (this directory) and `..` (its parent).

## Seeing structure — `tree`

`ls` shows one directory at a time. To see a whole branch at a glance — folders
within folders — use `tree`:

```{command-card} tree
```

In [13]:
tree -L 2 data

data


├── README.md


├── inputs


│   ├── geo-opt.inp


│   ├── production-md.inp


...

│   └── sn2-neb.ener


└── trajectories


    ├── lj38-optimization.xyz


    └── lj38-relaxed.xyz


5 directories, 10 files


That is exactly the schematic from the top of the notebook, drawn by the machine
from the real folders. `-L 2` kept it to two levels deep; drop it on a big tree
and you may get more than you bargained for.

## Tab completion — the navigation superpower

Everything above you can do with careful typing. This last one you can only do by
*pressing a key*, so it cannot be shown in a printed cell — but it is the single
biggest speed-up in the whole notebook, so do not skip it.

Start typing a command or a path and press **Tab**: the shell completes it for you.
Type `cd data/tr` then Tab, and the shell fills in `cd data/trajectories/`. Press
Tab **twice** and it lists all the options that match. It cuts typos, saves
keystrokes, and means you never again misspell a long filename like
`lj38-optimization.xyz`.

```{admonition} Practise this in your terminal
:class: tip
Tab completion is interactive — there is nothing to render here, only something
to *do*. Open the [live terminal](https://mybinder.org/v2/gh/ramador09/bash-primer-public/HEAD?urlpath=shell/),
type `cd data/` and start pressing **Tab**, and feel how much faster moving around
becomes. This one habit will save you more time than any other in this course.
```

## Exercises

The pattern is the same as before: a task, a place for your answer, and an
automatic ✓. Each starts from the repo root, so paths line up.

### Exercise 1 (worked) — Where am I, what's here

Report where you are with `pwd`, list what is in the current directory with `ls`,
then list your home directory in full, including hidden files, with `ls -la ~`.

In [14]:
cd "$ROOT"

In [15]:
# (solution hidden on the public site)


/home/runner/work/bash-primer/bash-primer


CHANGELOG.md	   _config.yml	 environment.yml	   reference


CLAUDE.md	   _ext		 jupyter_server_config.py  references.bib


LICENSE-CODE	   _static	 manifest.yml		   requirements.txt


LICENSE-CONTENT    _toc.yml	 modulefiles		   robots.txt


...

drwxr-x---  15 runner runner 4096 Jun 12 02:56 .


drwxr-xr-x+  5 root   root   4096 Jun  7 22:22 ..


-rw-------   1 runner runner 1549 Jun 12 02:56 .bash_history


-rw-r--r--   1 runner runner  220 Mar 31  2024 .bash_logout


-rw-r--r--   1 runner runner   67 Jun  7 21:47 .bash_profile


In [16]:
check '[ -n "$(pwd)" ] && ls >/dev/null 2>&1' "pwd reports where you are and ls runs"

✓ pwd reports where you are and ls runs


### Exercise 2 (your turn) — Reach the data directory

Move into the `data/` directory with `cd`, then list it in long, human-readable
form with `ls -lh`. (Afterwards, `pwd` should end in `data`.)

In [17]:
cd "$ROOT"

In [18]:
# (solution hidden on the public site)


total 20K


-rw-r--r-- 1 runner runner 2.0K Jun  1  2023 README.md


drwxr-xr-x 2 runner runner 4.0K Mar  1  2023 inputs


drwxr-xr-x 2 runner runner 4.0K Feb  1  2023 logs


drwxr-xr-x 2 runner runner 4.0K May  1  2023 results


drwxr-xr-x 2 runner runner 4.0K Apr  1  2023 trajectories


In [19]:
check '[ "$(basename "$PWD")" = "data" ]' "you are now standing in the data directory"

✓ you are now standing in the data directory


### Exercise 3 (your turn) — Same file, two paths

Point at the file `data/results/pt-slab.xyz` with a **relative** path, then print
its **absolute** path (hint: `$(pwd)` is the absolute path of where you are).

In [20]:
cd "$ROOT"

In [21]:
# (solution hidden on the public site)


data/results/pt-slab.xyz


/home/runner/work/bash-primer/bash-primer/data/results/pt-slab.xyz


In [22]:
check '[ -f "$(pwd)/data/results/pt-slab.xyz" ]' "the absolute path resolves to the same real file"

✓ the absolute path resolves to the same real file


### Exercise 4 (worked) — Visualize the tree

Draw the `data/` tree two levels deep with `tree -L 2`.

In [23]:
cd "$ROOT"

In [24]:
# (solution hidden on the public site)


data


├── README.md


├── inputs


│   ├── geo-opt.inp


│   ├── production-md.inp


...

│   └── sn2-neb.ener


└── trajectories


    ├── lj38-optimization.xyz


    └── lj38-relaxed.xyz


5 directories, 10 files


In [25]:
check 'tree -L 2 data | grep -q "trajectories"' "the tree shows the trajectories/ subdirectory"

✓ the tree shows the trajectories/ subdirectory


### Exercise 5 (your turn) — The right `ls` flags

List `data/` with one combined set of flags so that you see **hidden files**,
**human-readable sizes**, and entries **sorted newest-first by time**. Which entry
is newest? (The flags `-a`, `-h`, `-t`, and `-l` combine into one.)

In [26]:
cd "$ROOT"

In [27]:
# (solution hidden on the public site)


total 32K


drwxr-xr-x 13 runner runner 4.0K Jun 12 02:56 ..


drwxr-xr-x  6 runner runner 4.0K Jun 12 02:56 .


-rw-r--r--  1 runner runner  460 Jun 15  2023 .dataset-notes


-rw-r--r--  1 runner runner 2.0K Jun  1  2023 README.md


drwxr-xr-x  2 runner runner 4.0K May  1  2023 results


drwxr-xr-x  2 runner runner 4.0K Apr  1  2023 trajectories


drwxr-xr-x  2 runner runner 4.0K Mar  1  2023 inputs


drwxr-xr-x  2 runner runner 4.0K Feb  1  2023 logs


In [28]:
check 'ls -laht data | grep -q ".dataset-notes"' "the listing reveals the hidden .dataset-notes (and -t puts the newest first)"

✓ the listing reveals the hidden .dataset-notes (and -t puts the newest first)


### Exercise 6 (terminal-only) — Tab completion

This one has no ✓ — it is interactive. In the
[live terminal](https://mybinder.org/v2/gh/ramador09/bash-primer-public/HEAD?urlpath=shell/),
type `cd data/` then the first letter or two of a subdirectory, and press **Tab**.
Watch the shell finish the name. Then try it on a long filename inside
`trajectories/`. That reflex — type a little, press Tab — is worth building now.

## Outlook

You can now find your way to any file in the tree: you know where you are, how to
move, how to name a place two different ways, and how to see what is around you.
What you cannot yet do is look *inside* a file or move one around. That is the next
notebook — `cat`, `less`, `head`, `tail` to peek at contents, and `cp`, `mv`,
`rm`, `mkdir` to organize them — and it is the last stretch of groundwork before
the text-extraction heart of the course in Part II.

```{compendium-new}
```

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practice here&rdquo; box in any
    section to run everything yourself — nothing to install. The published
    notebooks ship <b>without worked solutions</b>; if you would like the
    reference solutions — to teach from or to check your own work — get in
    touch: <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>